In [1]:
#  imports and paths
import os, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_fscore_support
)

import matplotlib.pyplot as plt
import seaborn as sns

# PROJECT PATHS
PROJECT_ROOT = Path("/Users/syedadnanahmad/Downloads/AI_TUMOUR_detection")
EMB_ROOT     = PROJECT_ROOT / "embeddings"
IDX_CSV      = PROJECT_ROOT / "notebooks" / "slide_index.csv"
MODEL_DIR    = PROJECT_ROOT / "models"
RESULTS_DIR  = PROJECT_ROOT / "results"

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# DEVICE
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

# Load slide index CSV
df_index = pd.read_csv(IDX_CSV)
print(df_index.head())


Device: mps
   split   class   slide_id  \
0  train  mdoscc  o-3-00-01   
1  train  mdoscc  o-3-00-02   
2  train  mdoscc  o-3-00-03   
3  train  mdoscc  o-3-00-04   
4  train  mdoscc  o-3-00-05   

                                          slide_path  n_patches  \
0  /Users/syedadnanahmad/Downloads/AI_Tumour_dete...         77   
1  /Users/syedadnanahmad/Downloads/AI_Tumour_dete...         71   
2  /Users/syedadnanahmad/Downloads/AI_Tumour_dete...         73   
3  /Users/syedadnanahmad/Downloads/AI_Tumour_dete...         71   
4  /Users/syedadnanahmad/Downloads/AI_Tumour_dete...         72   

                                         patch_paths  
0  ['/Users/syedadnanahmad/Downloads/AI_Tumour_de...  
1  ['/Users/syedadnanahmad/Downloads/AI_Tumour_de...  
2  ['/Users/syedadnanahmad/Downloads/AI_Tumour_de...  
3  ['/Users/syedadnanahmad/Downloads/AI_Tumour_de...  
4  ['/Users/syedadnanahmad/Downloads/AI_Tumour_de...  


In [2]:
# Cell 2: Prepare Stage-1 binary dataset
# Classes: NORMAL = 0, OSCC = 1
OSCC_CLASSES = ["wdoscc", "mdoscc", "pdoscc"]
NORMAL_CLASS = ["normal"]

def make_stage1_df(df):
    mask = df["class"].isin(OSCC_CLASSES + NORMAL_CLASS)
    df2 = df[mask].copy().reset_index(drop=True)

    df2["binary_label"] = df2["class"].apply(lambda c: 1 if c in OSCC_CLASSES else 0)
    return df2

df_stage1 = make_stage1_df(df_index)

print("Stage-1 Data Counts:")
print(df_stage1.groupby(["split", "binary_label"]).size())


Stage-1 Data Counts:
split  binary_label
test   0                14
       1               105
train  0                14
       1               104
val    0                14
       1               104
dtype: int64


In [3]:
# Cell 3: Binary dataset class
class BinarySlideDataset(Dataset):
    def __init__(self, df, split):
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.items = []

        for _, row in self.df.iterrows():
            slide = row["slide_id"]
            cls   = row["class"]
            label = int(row["binary_label"])

            emb_path  = EMB_ROOT / split / cls / f"{slide}.npy"
            meta_path = EMB_ROOT / split / cls / f"{slide}_meta.json"

            if emb_path.exists():
                self.items.append({
                    "emb_path": emb_path,
                    "label": label,
                    "slide_id": slide,
                    "class": cls
                })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        emb = torch.from_numpy(np.load(it["emb_path"]).astype(np.float32))
        label = torch.tensor(it["label"], dtype=torch.long)
        return {"emb": emb, "label": label, "slide_id": it["slide_id"], "class": it["class"]}


In [4]:
# Cell 4: Create datasets & loaders
train_ds = BinarySlideDataset(df_stage1, "train")
val_ds   = BinarySlideDataset(df_stage1, "val")
test_ds  = BinarySlideDataset(df_stage1, "test")

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=lambda x: x[0])
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])

print("Train slides:", len(train_ds))
print("Val slides:", len(val_ds))
print("Test slides:", len(test_ds))

# Compute class weights
from collections import Counter
counts = Counter([it["label"] for it in train_ds.items])
total = sum(counts.values())
weights = [ total / (2 * counts[i]) for i in range(2) ]
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)


Train slides: 118
Val slides: 118
Test slides: 119


In [5]:
# Cell 5: K=20 Top-K MIL for binary classification
class BinaryTopKMIL(nn.Module):
    def __init__(self, emb_dim=512, hidden_dim=256, k=20):
        super().__init__()
        self.k = k

        self.V = nn.Linear(emb_dim, hidden_dim)
        self.U = nn.Linear(emb_dim, hidden_dim)
        self.w = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Sequential(
            nn.Linear(emb_dim, emb_dim//2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(emb_dim//2, 2)
        )

    def forward(self, H):
        Vh = torch.tanh(self.V(H))
        Uh = torch.sigmoid(self.U(H))
        A  = self.w(Vh * Uh).squeeze(1)

        K = min(self.k, H.shape[0])
        top_vals, top_idx = torch.topk(A, K)

        H_top = H[top_idx]
        A_top = torch.softmax(top_vals, dim=0)

        agg = torch.sum(A_top.unsqueeze(1) * H_top, dim=0)
        logits = self.classifier(agg)
        return logits.unsqueeze(0)


In [6]:
# Cell 6: Training loop for Stage-1
def train_epoch(model, loader, optimizer):
    model.train()
    losses, y_true, y_pred = [], [], []

    for batch in loader:
        emb = batch["emb"].to(device)
        label = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(emb)
        loss = criterion(logits, label.unsqueeze(0))
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        p = torch.softmax(logits, dim=1).detach().cpu().numpy()[0]
        y_pred.append(np.argmax(p))
        y_true.append(int(label))

    avg_loss = np.mean(losses)
    acc = accuracy_score(y_true, y_pred)
    f1  = precision_recall_fscore_support(y_true, y_pred, average='macro')[2]
    return avg_loss, acc, f1

def val_epoch(model, loader):
    model.eval()
    losses, y_true, y_pred = [], [], []

    with torch.no_grad():
        for batch in loader:
            emb = batch["emb"].to(device)
            label = batch["label"].to(device)

            logits = model(emb)
            loss = criterion(logits, label.unsqueeze(0))
            losses.append(loss.item())

            p = torch.softmax(logits, dim=1).cpu().numpy()[0]
            y_pred.append(np.argmax(p))
            y_true.append(int(label))

    avg_loss = np.mean(losses)
    acc = accuracy_score(y_true, y_pred)
    f1  = precision_recall_fscore_support(y_true, y_pred, average='macro')[2]
    return avg_loss, acc, f1


In [7]:
# Cell 7: Train Stage-1 Binary Model

model_stage1 = BinaryTopKMIL(k=20).to(device)
optimizer = torch.optim.AdamW(model_stage1.parameters(), lr=1e-4, weight_decay=1e-5)

best_f1 = -1
patience = 5
no_imp = 0

for epoch in range(1, 25):
    t0 = time.time()
    tr_loss, tr_acc, tr_f1 = train_epoch(model_stage1, train_loader, optimizer)
    va_loss, va_acc, va_f1 = val_epoch(model_stage1, val_loader)

    print(f"Epoch {epoch}: Train F1={tr_f1:.3f}, Val F1={va_f1:.3f}")

    if va_f1 > best_f1:
        best_f1 = va_f1
        no_imp = 0
        torch.save(model_stage1.state_dict(), MODEL_DIR/"stage1_binary_best.pth")
        print("Saved best model.")
    else:
        no_imp += 1
        if no_imp >= patience:
            print("Early stopping.")
            break


/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 1: Train F1=0.508, Val F1=0.468
Saved best model.


/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 2: Train F1=0.468, Val F1=0.468


/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 5: Train F1=0.468, Val F1=0.468


/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Epoch 6: Train F1=0.468, Val F1=0.468
Early stopping.


/opt/anaconda3/envs/oral_msc/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
# Cell 8: Test evaluation
model_stage1 = BinaryTopKMIL(k=20).to(device)
model_stage1.load_state_dict(torch.load(MODEL_DIR/"stage1_binary_best.pth", map_location=device))
model_stage1.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        emb = batch["emb"].to(device)
        label = int(batch["label"])
        logits = model_stage1(emb)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]
        y_pred.append(np.argmax(p))
        y_true.append(label)

print("\n=== Stage-1 TEST REPORT ===")
print(classification_report(y_true, y_pred, target_names=["normal","oscc"], zero_division=0))



=== Stage-1 TEST REPORT ===
              precision    recall  f1-score   support

      normal       0.00      0.00      0.00        14
        oscc       0.88      1.00      0.94       105

    accuracy                           0.88       119
   macro avg       0.44      0.50      0.47       119
weighted avg       0.78      0.88      0.83       119

